# Full Dataset YOLO Training and Leave-One-Out Evaluation

## Purpose

This notebook trains a YOLO object detection model using the full expertized dataset, then uses that trained model as the starting point for leave-one-video-out evaluation on the separated IronHorse dataset.

The notebook has two main sections:

1. **Full expertized dataset training**
   - Loads the complete expertized dataset from Google Drive.
   - Flattens the dataset folder structure.
   - Converts LabelMe-style JSON annotations into YOLO format.
   - Merges selected leaf labels into one class: `unhealthy_leaf`.
   - Trains a YOLOv8 model on the full dataset.

2. **Leave-one-video-out IronHorse evaluation**
   - Loads the separated IronHorse dataset.
   - Creates one fold per video.
   - Holds out one video for validation and trains on the remaining videos.
   - Starts each fold from the model trained on the full expertized dataset.
   - Evaluates each fold and creates visual review images comparing ground-truth labels with predictions.

## What this notebook does

This notebook performs the following steps:

1. **Installs Ultralytics**
   Installs the YOLO training library used throughout the notebook.

2. **Mounts Google Drive**
   Connects Colab to Google Drive so datasets and outputs can be accessed.

3. **Extracts the complete expertized dataset**
   Unzips `Complete_Data_set.zip` into the Colab runtime.

4. **Flattens the expertized dataset**
   Copies all images and JSON annotation files into one folder so they are easier to process.

5. **Converts JSON annotations into YOLO format**
   Converts rectangle annotations from LabelMe JSON files into YOLO `.txt` labels.

6. **Merges multiple labels into one class**
   The labels `FD leaf`, `confounding leaf`, and `ESCA leaf` are all converted into one YOLO class: `unhealthy_leaf`.

7. **Drops unused labels**
   Labels such as `symptomatic bunch`, `facteur confondant +`, `autre`, `symptomatic shoot`, and `*` are ignored.

8. **Creates a train-only YOLO dataset**
   The notebook writes a YOLO dataset where both `train` and `val` point to the training image folder. This means the first training section is focused on training from the full dataset, not measuring a separate validation split.

9. **Trains a YOLOv8 model**
   Trains `yolov8.pt` on the full expertized dataset.

10. **Extracts the separated IronHorse dataset**
    Unzips `SeparatedDataSet.zip`, which is expected to be organized by video folders.

11. **Creates leave-one-video-out folds**
    Each video folder becomes a validation fold. The held-out video is used for validation, and the remaining videos are used for training.

12. **Trains one model per fold**
    Each fold starts from the full-dataset model weights and continues training on the IronHorse fold data.

13. **Validates selected fold models**
    Evaluates trained fold models and prints precision, recall, mAP50, and mAP50-95.

14. **Creates ground-truth-versus-prediction review images**
    Draws ground-truth boxes and predicted boxes on the same images for visual inspection.

15. **Copies review outputs to Google Drive**
    Saves visual review folders to Google Drive so they can be reviewed later outside Colab.

## How to use this notebook
###Part 1: Train on the full expertized dataset

1. Make sure the complete expertized dataset ZIP file is stored in Google Drive.
2. Check or update this path:

   ```python
   zip_path = "/content/drive/MyDrive/Complete_Data_set.zip"
   ```

3. Run the setup cells to: Install Ultralytics, mount Google Drive, extract dataset, flatten folder structure, convert JSON labels into YOLO labels, and create the YOLO data.yaml file

4. Run the Training cell (Can change the size of the model to see if performance improves)

```python
model = YOLO("yolov8l.pt") #options: yolov8s.pt, yolov8m.pt, yolov8n.pt, other YOLO models
```

###Part 2: Train and evaluate leave-one-video out folds

1. Make sure the separated IronHorse dataset ZIP file is stored in Google Drive
2. Check or update this path:

   ```python
   zip_path = "/content/drive/MyDrive/SeparatedDataSet.zip"
   ```
3. Run the fold creation section to: extract the separated data set, find video folders, create one fold per video, and write a data.yaml file for each fold
4. Run the fold training loop
5. Validate individual folds by checking the model path and fold YAML file:

   ```python
   model = YOLO("/content/yolo_folds/fold_Video 011/weights/best.pt")
   metrics = model.val(data="/content/folds/fold_Video 011/data.yaml")
   ```

6. Review the printed metrics: Precision, Recall, mAP50, mAP50-95
7. Run the fold creation cells to get a folder of each model predictions on the images
8. Creates a ZIP file to download and view on a local system

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Complete_Data_set.zip"

extract_dir = "/content/dataset_raw"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)
print("Top-level contents:", os.listdir(extract_dir)[:10])

Extracted to: /content/dataset_raw
Top-level contents: ['Complete_Data_set']


In [ ]:
import shutil

input_root = "/content/dataset_raw/Complete_Data_set/Complete_Data_set"
flat_dir = "/content/dataset_flat"
os.makedirs(flat_dir, exist_ok=True)

copied = 0

for root, dirs, files in os.walk(input_root):
    rel_root = os.path.relpath(root, input_root)
    if rel_root == ".":
        folder_prefix = "root"
    else:
        folder_prefix = rel_root.replace(os.sep, "_")

    for f in files:
        lower = f.lower()
        if lower.endswith(".jpg") or lower.endswith(".jpeg") or lower.endswith(".png") or lower.endswith(".json"):
            old_path = os.path.join(root, f)
            new_name = f"{folder_prefix}_{f}"
            new_path = os.path.join(flat_dir, new_name)

            shutil.copy2(old_path, new_path)
            copied += 1

print("Total copied files:", copied)
print("Flat dataset directory:", flat_dir)
print("Sample files:", sorted(os.listdir(flat_dir))[:20])

Total copied files: 1174
Flat dataset directory: /content/dataset_flat
Sample files: ['CF21_im_00020.jpg', 'CF21_im_00020.json', 'CF21_im_00024.jpg', 'CF21_im_00024.json', 'CF21_im_00025.jpg', 'CF21_im_00025.json', 'CF21_im_00027.jpg', 'CF21_im_00027.json', 'CF21_im_00028.jpg', 'CF21_im_00028.json', 'CF21_im_00029.jpg', 'CF21_im_00029.json', 'CF21_im_00031.jpg', 'CF21_im_00031.json', 'CF21_im_00035.jpg', 'CF21_im_00035.json', 'CF21_im_00036.jpg', 'CF21_im_00036.json', 'CF21_im_00037.jpg', 'CF21_im_00037.json']


In [ ]:
from pathlib import Path
import json
from PIL import Image

src = Path("dataset_flat")
dst = Path("dataset_yolo_unhealthy_leaf")

images_train = dst / "images" / "train"
labels_train = dst / "labels" / "train"

images_train.mkdir(parents=True, exist_ok=True)
labels_train.mkdir(parents=True, exist_ok=True)

# Original LabelMe labels that should become the new YOLO class
merge_into_unhealthy_leaf = {
    "FD leaf",
    "confounding leaf",
    "ESCA leaf",
}

# Original LabelMe labels to ignore completely
drop_labels = {
    "symptomatic bunch",
    "facteur confondant +",
    "autre",
    "symptomatic shoot",
    "*"
}

class_names = ["unhealthy_leaf"]

image_extensions = [".jpg", ".jpeg", ".png"]

for img_path in src.iterdir():
    if img_path.suffix.lower() not in image_extensions:
        continue

    json_path = img_path.with_suffix(".json")

    shutil.copy2(img_path, images_train / img_path.name)

    with Image.open(img_path) as im:
        img_w, img_h = im.size

    yolo_lines = []

    if json_path.exists():
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for shape in data.get("shapes", []):
            label = shape["label"]
            shape_type = shape.get("shape_type")

            if shape_type != "rectangle":
                print(f"Skipping non-rectangle shape in {json_path.name}: {shape_type}")
                continue

            if label in drop_labels:
                continue

            if label not in merge_into_unhealthy_leaf:
                raise ValueError(
                    f"Unknown label '{label}' in {json_path.name}. "
                    "Add it to merge_into_unhealthy_leaf or drop_labels."
                )

            # Both FD leaf and confounding leaf become class 0: unhealthy_leaf
            class_id = 0

            points = shape["points"]
            x1, y1 = points[0]
            x2, y2 = points[1]

            x_min = max(0, min(x1, x2))
            x_max = min(img_w, max(x1, x2))
            y_min = max(0, min(y1, y2))
            y_max = min(img_h, max(y1, y2))

            box_w = x_max - x_min
            box_h = y_max - y_min

            if box_w <= 0 or box_h <= 0:
                print(f"Skipping invalid box in {json_path.name}")
                continue

            x_center = (x_min + x_max) / 2 / img_w
            y_center = (y_min + y_max) / 2 / img_h
            norm_w = box_w / img_w
            norm_h = box_h / img_h

            yolo_lines.append(
                f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}"
            )

    label_path = labels_train / f"{img_path.stem}.txt"
    label_path.write_text("\n".join(yolo_lines), encoding="utf-8")

yaml_text = f"""path: {dst.resolve()}
train: images/train
val: images/train

names:
  0: unhealthy_leaf
"""

(dst / "data.yaml").write_text(yaml_text, encoding="utf-8")

print(f"YOLO train-only dataset created at: {dst.resolve()}")
print("Class 0: unhealthy_leaf")
print("Dropped labels:", drop_labels)

Skipping non-rectangle shape in UB20(2)_im_00030.json: linestrip
Skipping non-rectangle shape in UB20(2)_im_00030.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Skipping non-rectangle shape in CS20_im_00092.json: linestrip
Sk

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8l.pt")

model.train(
    data="/content/dataset_yolo_unhealthy_leaf/data.yaml",
    epochs= 50,
    imgsz=1280,
    batch=16,
    workers=8,
    patience = 30,
    cache=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_yolo_unhealthy_leaf/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b3282284f80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
zip_path = "/content/drive/MyDrive/SeparatedDataSet.zip"

extract_dir = "/content/dataset_raw"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)
print("Top-level contents:", os.listdir(extract_dir)[:10])

Extracted to: /content/dataset_raw
Top-level contents: ['Complete_Data_set', 'Video 012', 'Video 011', 'Video 014', 'video 0']


In [ ]:
from pathlib import Path
import yaml

SOURCE_ROOT = Path("/content/dataset_raw")
OUTPUT_ROOT = Path("/content//folds")

CLASS_NAMES = ["unhealthly_leaf"]
COPY_FILES = True  # True = copy files, False = symlink files

# =========================
# Helper functions
# =========================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def get_video_folders(source_root):
    """Return subfolders that contain train/images and train/labels."""
    video_folders = []

    for folder in sorted(source_root.iterdir()):
        if not folder.is_dir():
            continue

        img_dir = folder / "train" / "images"
        lbl_dir = folder / "train" / "labels"

        if img_dir.exists() and lbl_dir.exists():
            video_folders.append(folder)

    return video_folders


def link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        return

    if COPY_FILES:
        shutil.copy2(src, dst)
    else:
        dst.symlink_to(src)


def add_video_to_split(video_folder, fold_root, split):
    """
    Adds one video's images and labels to a YOLO split.
    split should be 'train' or 'val'.
    """
    src_img_dir = video_folder / "train" / "images"
    src_lbl_dir = video_folder / "train" / "labels"

    dst_img_dir = fold_root / "images" / split
    dst_lbl_dir = fold_root / "labels" / split

    for img_path in sorted(src_img_dir.iterdir()):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue

        # Prefix filenames with video folder name to avoid duplicate names across videos
        new_img_name = f"{video_folder.name}_{img_path.name}"
        dst_img_path = dst_img_dir / new_img_name

        link_or_copy(img_path, dst_img_path)

        # YOLO label should have same stem as image, with .txt extension
        src_label_path = src_lbl_dir / f"{img_path.stem}.txt"
        new_label_name = f"{video_folder.name}_{img_path.stem}.txt"
        dst_label_path = dst_lbl_dir / new_label_name

        if src_label_path.exists():
            link_or_copy(src_label_path, dst_label_path)
        else:
            # Optional: create empty label file for images with no objects
            dst_label_path.parent.mkdir(parents=True, exist_ok=True)
            dst_label_path.touch()


def write_data_yaml(fold_root, class_names):
    data = {
        "path": str(fold_root),
        "train": "images/train",
        "val": "images/val",
        "names": {i: name for i, name in enumerate(class_names)}
    }

    with open(fold_root / "data.yaml", "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)


# =========================
# Main leave-one-out split
# =========================
video_folders = get_video_folders(SOURCE_ROOT)

print(f"Found {len(video_folders)} video folders:")
for v in video_folders:
    print(" -", v.name)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for val_video in video_folders:
    fold_root = OUTPUT_ROOT / f"fold_{val_video.name}"

    print(f"\nCreating fold: {fold_root.name}")
    print(f"Validation video: {val_video.name}")

    # Optional: remove old fold before recreating
    if fold_root.exists():
        shutil.rmtree(fold_root)

    # Create train/val folders
    for split in ["train", "val"]:
        (fold_root / "images" / split).mkdir(parents=True, exist_ok=True)
        (fold_root / "labels" / split).mkdir(parents=True, exist_ok=True)

    # Add videos
    for video in video_folders:
        if video == val_video:
            add_video_to_split(video, fold_root, "val")
        else:
            add_video_to_split(video, fold_root, "train")

    write_data_yaml(fold_root, CLASS_NAMES)


Found 4 video folders:
 - Video 011
 - Video 012
 - Video 014
 - video 0

Creating fold: fold_Video 011
Validation video: Video 011

Creating fold: fold_Video 012
Validation video: Video 012

Creating fold: fold_Video 014
Validation video: Video 014

Creating fold: fold_video 0
Validation video: video 0


In [ ]:
from ultralytics import YOLO

folds_root = Path("/content/folds")
results_root = "/content/yolo_folds/"

for fold_root in sorted(folds_root.iterdir()):
    if not fold_root.is_dir():
        continue

    data_yaml = fold_root / "data.yaml"

    if not data_yaml.exists():
        print(f"Skipping {fold_root.name}: no data.yaml found")
        continue

    print(f"\nTraining fold: {fold_root.name}")

    # Fresh model for each fold
    model = YOLO("/content/runs/detect/train/weights/best.pt")

    model.train(
        data=str(data_yaml),
        epochs=100,
        patience=15,
        imgsz=1280,
        batch=16,
        # freeze = 5,
        # lr0 = 0.001,
        project=results_root,
        name=fold_root.name,
    )


Training fold: fold_Video 011
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/folds/fold_Video 011/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold_Video 011

In [ ]:
model = YOLO("/content/yolo_folds/fold_Video 011/weights/best.pt")

metrics = model.val(data="/content/folds/fold_Video 011/data.yaml")

print("Overall metrics")
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 113 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3502.1±928.8 MB/s, size: 507.9 KB)
val: Scanning /content/folds/fold_Video 011/labels/val.cache... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 5.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.2s/it 1.2s
                   all         14         40      0.265      0.375      0.243      0.125
Speed: 2.9ms preprocess, 14.0ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/runs/detect/val-5
Overall metrics
Precision: 0.2647968324396893
Recall: 0.375
mAP50: 0.24302608136124998
mAP50-95: 0.12482578161449391


In [ ]:
model = YOLO("/content/yolo_folds/fold_Video 012/weights/best.pt")

metrics = model.val(data="/content/folds/fold_Video 012/data.yaml")

print("Overall metrics")
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 113 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2533.0±790.1 MB/s, size: 575.4 KB)
val: Scanning /content/folds/fold_Video 012/labels/val.cache... 53 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 53/53 22.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.2it/s 3.2s
                   all         53        130      0.428      0.477       0.36      0.116
Speed: 15.3ms preprocess, 14.2ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detect/val-6
Overall metrics
Precision: 0.4280476989090937
Recall: 0.47692307692307695
mAP50: 0.35952114036062754
mAP50-95: 0.11642255688750056


In [ ]:
model = YOLO("/content/yolo_folds/fold_Video 014/weights/best.pt")
metrics = model.val(data="/content/folds/fold_Video 014/data.yaml")

print("Overall metrics")
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3148.9±228.5 MB/s, size: 703.5 KB)
val: Scanning /content/folds/fold_Video 014/labels/val... 45 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 45/45 1.6Kit/s 0.0s
val: New cache created: /content/folds/fold_Video 014/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.0it/s 2.9s
                   all         45         96      0.544      0.447      0.422      0.148
Speed: 9

In [ ]:
model = YOLO("/content/yolo_folds/fold_video 0/weights/best.pt")

metrics = model.val(data="/content/folds/fold_video 0/data.yaml")

print("Overall metrics")
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Model summary (fused): 113 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3555.0±445.6 MB/s, size: 2061.4 KB)
val: Scanning /content/folds/fold_video 0/labels/val.cache... 129 images, 29 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 129/129 45.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.1it/s 4.3s
                   all        129        288      0.319       0.25      0.178     0.0536
Speed: 5.4ms preprocess, 7.9ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/runs/detect/val-8
Overall metrics
Precision: 0.3192422805110697
Recall: 0.25
mAP50: 0.17797370695247108
mAP50-95: 0.05361226067892355


In [ ]:
import cv2
!pip install ultralytics -q

from ultralytics import YOLO


model_path = "/content/yolo_folds/fold_Video 011/weights/best.pt"   # change if needed
dataset_root = Path("/content/folds/fold_Video 011")        # change if needed
output_root = Path("/content/gt_vs_pred_fold_Video11")

# image folders
train_images_dir = dataset_root / "images" / "train"
val_images_dir   = dataset_root / "images" / "val"

# label folders
train_labels_dir = dataset_root / "labels" / "train"
val_labels_dir   = dataset_root / "labels" / "val"

# output folders
(output_root / "train").mkdir(parents=True, exist_ok=True)
(output_root / "val").mkdir(parents=True, exist_ok=True)

model_copy_path = output_root / "best.pt"
shutil.copy2(model_path, model_copy_path)

# class names
class_names = {0: "unhealthy_leaf"}

# colors in BGR for OpenCV
GT_COLOR = (0, 0, 255)      # red
PRED_COLOR = (0, 255, 255)    # yellow

# prediction settings
CONF_THRES = 0.30
IOU_THRES = 0.30

model = YOLO(model_path)

def yolo_to_xyxy(line, img_w, img_h):
    """
    Convert YOLO label line:
    class x_center y_center width height
    (normalized)
    to pixel xyxy
    """
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    cls_id = int(float(parts[0]))
    x_c = float(parts[1]) * img_w
    y_c = float(parts[2]) * img_h
    w = float(parts[3]) * img_w
    h = float(parts[4]) * img_h

    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)

    return cls_id, x1, y1, x2, y2

def draw_ground_truth(img, label_path):
    h, w = img.shape[:2]

    if not label_path.exists():
        return img

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parsed = yolo_to_xyxy(line, w, h)
        if parsed is None:
            continue

        cls_id, x1, y1, x2, y2 = parsed

        cv2.rectangle(img, (x1, y1), (x2, y2), GT_COLOR, 2)

        text = "GT"
        cv2.putText(
            img,
            text,
            (x1, max(y1 - 8, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            GT_COLOR,
            2
        )

    return img

def draw_predictions(img, result):
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        cv2.rectangle(img, (x1, y1), (x2, y2), PRED_COLOR, 2)

        # confidence only, no class label
        text = f"{conf:.2f}"
        cv2.putText(
            img,
            text,
            (x1, min(y2 + 20, img.shape[0] - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            PRED_COLOR,
            2
        )

    return img

def process_folder(images_dir, labels_dir, save_dir):
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_files = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in image_extensions])

    print(f"Processing {len(image_files)} images from {images_dir}")

    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"

        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Could not read: {img_path}")
            continue

        # draw ground truth first
        img = draw_ground_truth(img, label_path)

        # predict on this single image
        result = model.predict(
            source=str(img_path),
            conf=CONF_THRES,
            iou=IOU_THRES,
            save=False,
            verbose=False
        )[0]

        # draw predictions second
        img = draw_predictions(img, result)

        out_path = save_dir / img_path.name
        cv2.imwrite(str(out_path), img)

    print(f"Saved outputs to {save_dir}")

# process train and val
process_folder(train_images_dir, train_labels_dir, output_root / "train")
process_folder(val_images_dir, val_labels_dir, output_root / "val")

print(f"Done. Outputs saved under: {output_root}")

Processing 227 images from /content/folds/fold_Video 011/images/train
Saved outputs to /content/gt_vs_pred_fold_Video11/train
Processing 14 images from /content/folds/fold_Video 011/images/val
Saved outputs to /content/gt_vs_pred_fold_Video11/val
Done. Outputs saved under: /content/gt_vs_pred_fold_Video11


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/gt_vs_pred_fold_Video11")  # folder that contains train/ and val/
dst_root = Path("/content/drive/MyDrive/1Trash/Yolov8L/V011")

dst_root.mkdir(parents=True, exist_ok=True)

dst = dst_root / src.name

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print(f"Copied folder to: {dst}")

Copied folder to: /content/drive/MyDrive/1Trash/Yolov8L/V011/gt_vs_pred_fold_Video11


In [ ]:
model_path = "/content/yolo_folds/fold_Video 012/weights/best.pt"   # change if needed
dataset_root = Path("/content/folds/fold_Video 012")        # change if needed
output_root = Path("/content/gt_vs_pred_fold_Video12")

# image folders
train_images_dir = dataset_root / "images" / "train"
val_images_dir   = dataset_root / "images" / "val"

# label folders
train_labels_dir = dataset_root / "labels" / "train"
val_labels_dir   = dataset_root / "labels" / "val"

# output folders
(output_root / "train").mkdir(parents=True, exist_ok=True)
(output_root / "val").mkdir(parents=True, exist_ok=True)

model_copy_path = output_root / "best.pt"
shutil.copy2(model_path, model_copy_path)

# class names
class_names = {0: "unhealthy_leaf"}

# colors in BGR for OpenCV
GT_COLOR = (0, 0, 255)      # red
PRED_COLOR = (0, 255, 255)    # yellow

# prediction settings
CONF_THRES = 0.30
IOU_THRES = 0.30

model = YOLO(model_path)

def yolo_to_xyxy(line, img_w, img_h):
    """
    Convert YOLO label line:
    class x_center y_center width height
    (normalized)
    to pixel xyxy
    """
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    cls_id = int(float(parts[0]))
    x_c = float(parts[1]) * img_w
    y_c = float(parts[2]) * img_h
    w = float(parts[3]) * img_w
    h = float(parts[4]) * img_h

    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)

    return cls_id, x1, y1, x2, y2

def draw_ground_truth(img, label_path):
    h, w = img.shape[:2]

    if not label_path.exists():
        return img

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parsed = yolo_to_xyxy(line, w, h)
        if parsed is None:
            continue

        cls_id, x1, y1, x2, y2 = parsed

        cv2.rectangle(img, (x1, y1), (x2, y2), GT_COLOR, 2)

        text = "GT"
        cv2.putText(
            img,
            text,
            (x1, max(y1 - 8, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            GT_COLOR,
            2
        )

    return img

def draw_predictions(img, result):
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        cv2.rectangle(img, (x1, y1), (x2, y2), PRED_COLOR, 2)

        # confidence only, no class label
        text = f"{conf:.2f}"
        cv2.putText(
            img,
            text,
            (x1, min(y2 + 20, img.shape[0] - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            PRED_COLOR,
            2
        )

    return img

def process_folder(images_dir, labels_dir, save_dir):
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_files = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in image_extensions])

    print(f"Processing {len(image_files)} images from {images_dir}")

    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"

        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Could not read: {img_path}")
            continue

        # draw ground truth first
        img = draw_ground_truth(img, label_path)

        # predict on this single image
        result = model.predict(
            source=str(img_path),
            conf=CONF_THRES,
            iou=IOU_THRES,
            save=False,
            verbose=False
        )[0]

        # draw predictions second
        img = draw_predictions(img, result)

        out_path = save_dir / img_path.name
        cv2.imwrite(str(out_path), img)

    print(f"Saved outputs to {save_dir}")

# process train and val
process_folder(train_images_dir, train_labels_dir, output_root / "train")
process_folder(val_images_dir, val_labels_dir, output_root / "val")

print(f"Done. Outputs saved under: {output_root}")

Processing 188 images from /content/folds/fold_Video 012/images/train
Saved outputs to /content/gt_vs_pred_fold_Video12/train
Processing 53 images from /content/folds/fold_Video 012/images/val
Saved outputs to /content/gt_vs_pred_fold_Video12/val
Done. Outputs saved under: /content/gt_vs_pred_fold_Video12


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/gt_vs_pred_fold_Video12")  # folder that contains train/ and val/
dst_root = Path("/content/drive/MyDrive/1Trash/Yolov8L/V012/")

dst_root.mkdir(parents=True, exist_ok=True)

dst = dst_root / src.name

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print(f"Copied folder to: {dst}")

Copied folder to: /content/drive/MyDrive/1Trash/Yolov8L/V012/gt_vs_pred_fold_Video12


In [ ]:
model_path = "/content/yolo_folds/fold_Video 014/weights/best.pt"   # change if needed
dataset_root = Path("/content/folds/fold_Video 014")        # change if needed
output_root = Path("/content/gt_vs_pred_fold_Video14")

# image folders
train_images_dir = dataset_root / "images" / "train"
val_images_dir   = dataset_root / "images" / "val"

# label folders
train_labels_dir = dataset_root / "labels" / "train"
val_labels_dir   = dataset_root / "labels" / "val"

# output folders
(output_root / "train").mkdir(parents=True, exist_ok=True)
(output_root / "val").mkdir(parents=True, exist_ok=True)

model_copy_path = output_root / "best.pt"
shutil.copy2(model_path, model_copy_path)

# class names
class_names = {0: "unhealthy_leaf"}

# colors in BGR for OpenCV
GT_COLOR = (0, 0, 255)      # red
PRED_COLOR = (0, 255, 255)    # yellow

# prediction settings
CONF_THRES = 0.30
IOU_THRES = 0.30

model = YOLO(model_path)

def yolo_to_xyxy(line, img_w, img_h):
    """
    Convert YOLO label line:
    class x_center y_center width height
    (normalized)
    to pixel xyxy
    """
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    cls_id = int(float(parts[0]))
    x_c = float(parts[1]) * img_w
    y_c = float(parts[2]) * img_h
    w = float(parts[3]) * img_w
    h = float(parts[4]) * img_h

    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)

    return cls_id, x1, y1, x2, y2

def draw_ground_truth(img, label_path):
    h, w = img.shape[:2]

    if not label_path.exists():
        return img

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parsed = yolo_to_xyxy(line, w, h)
        if parsed is None:
            continue

        cls_id, x1, y1, x2, y2 = parsed

        cv2.rectangle(img, (x1, y1), (x2, y2), GT_COLOR, 2)

        text = "GT"
        cv2.putText(
            img,
            text,
            (x1, max(y1 - 8, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            GT_COLOR,
            2
        )

    return img

def draw_predictions(img, result):
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        cv2.rectangle(img, (x1, y1), (x2, y2), PRED_COLOR, 2)

        # confidence only, no class label
        text = f"{conf:.2f}"
        cv2.putText(
            img,
            text,
            (x1, min(y2 + 20, img.shape[0] - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            PRED_COLOR,
            2
        )

    return img

def process_folder(images_dir, labels_dir, save_dir):
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_files = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in image_extensions])

    print(f"Processing {len(image_files)} images from {images_dir}")

    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"

        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Could not read: {img_path}")
            continue

        # draw ground truth first
        img = draw_ground_truth(img, label_path)

        # predict on this single image
        result = model.predict(
            source=str(img_path),
            conf=CONF_THRES,
            iou=IOU_THRES,
            save=False,
            verbose=False
        )[0]

        # draw predictions second
        img = draw_predictions(img, result)

        out_path = save_dir / img_path.name
        cv2.imwrite(str(out_path), img)

    print(f"Saved outputs to {save_dir}")

# process train and val
process_folder(train_images_dir, train_labels_dir, output_root / "train")
process_folder(val_images_dir, val_labels_dir, output_root / "val")

print(f"Done. Outputs saved under: {output_root}")

Processing 196 images from /content/folds/fold_Video 014/images/train
Saved outputs to /content/gt_vs_pred_fold_Video14/train
Processing 45 images from /content/folds/fold_Video 014/images/val
Saved outputs to /content/gt_vs_pred_fold_Video14/val
Done. Outputs saved under: /content/gt_vs_pred_fold_Video14


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/gt_vs_pred_fold_Video14")  # folder that contains train/ and val/
dst_root = Path("/content/drive/MyDrive/1Trash/Yolov8L/V014")

dst_root.mkdir(parents=True, exist_ok=True)

dst = dst_root / src.name

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print(f"Copied folder to: {dst}")

Copied folder to: /content/drive/MyDrive/1Trash/Yolov8L/V014/gt_vs_pred_fold_Video14


In [ ]:
model_path = "/content/yolo_folds/fold_video 0/weights/best.pt"   # change if needed
dataset_root = Path("/content/folds/fold_video 0")        # change if needed
output_root = Path("/content/gt_vs_pred_fold_Video0")

# image folders
train_images_dir = dataset_root / "images" / "train"
val_images_dir   = dataset_root / "images" / "val"

# label folders
train_labels_dir = dataset_root / "labels" / "train"
val_labels_dir   = dataset_root / "labels" / "val"

# output folders
(output_root / "train").mkdir(parents=True, exist_ok=True)
(output_root / "val").mkdir(parents=True, exist_ok=True)

model_copy_path = output_root / "best.pt"
shutil.copy2(model_path, model_copy_path)

# class names
class_names = {0: "unhealthy_leaf"}

# colors in BGR for OpenCV
GT_COLOR = (0, 0, 255)      # red
PRED_COLOR = (0, 255, 255)    # yellow

# prediction settings
CONF_THRES = 0.30
IOU_THRES = 0.30

model = YOLO(model_path)

def yolo_to_xyxy(line, img_w, img_h):
    """
    Convert YOLO label line:
    class x_center y_center width height
    (normalized)
    to pixel xyxy
    """
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    cls_id = int(float(parts[0]))
    x_c = float(parts[1]) * img_w
    y_c = float(parts[2]) * img_h
    w = float(parts[3]) * img_w
    h = float(parts[4]) * img_h

    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)

    return cls_id, x1, y1, x2, y2

def draw_ground_truth(img, label_path):
    h, w = img.shape[:2]

    if not label_path.exists():
        return img

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parsed = yolo_to_xyxy(line, w, h)
        if parsed is None:
            continue

        cls_id, x1, y1, x2, y2 = parsed

        cv2.rectangle(img, (x1, y1), (x2, y2), GT_COLOR, 2)

        text = "GT"
        cv2.putText(
            img,
            text,
            (x1, max(y1 - 8, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            GT_COLOR,
            2
        )

    return img

def draw_predictions(img, result):
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        cv2.rectangle(img, (x1, y1), (x2, y2), PRED_COLOR, 2)

        # confidence only, no class label
        text = f"{conf:.2f}"
        cv2.putText(
            img,
            text,
            (x1, min(y2 + 20, img.shape[0] - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            PRED_COLOR,
            2
        )

    return img

def process_folder(images_dir, labels_dir, save_dir):
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_files = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in image_extensions])

    print(f"Processing {len(image_files)} images from {images_dir}")

    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"

        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Could not read: {img_path}")
            continue

        # draw ground truth first
        img = draw_ground_truth(img, label_path)

        # predict on this single image
        result = model.predict(
            source=str(img_path),
            conf=CONF_THRES,
            iou=IOU_THRES,
            save=False,
            verbose=False
        )[0]

        # draw predictions second
        img = draw_predictions(img, result)

        out_path = save_dir / img_path.name
        cv2.imwrite(str(out_path), img)

    print(f"Saved outputs to {save_dir}")

# process train and val
process_folder(train_images_dir, train_labels_dir, output_root / "train")
process_folder(val_images_dir, val_labels_dir, output_root / "val")

print(f"Done. Outputs saved under: {output_root}")

Processing 112 images from /content/folds/fold_video 0/images/train
Saved outputs to /content/gt_vs_pred_fold_Video0/train
Processing 129 images from /content/folds/fold_video 0/images/val
Saved outputs to /content/gt_vs_pred_fold_Video0/val
Done. Outputs saved under: /content/gt_vs_pred_fold_Video0


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/gt_vs_pred_fold_Video0")  # folder that contains train/ and val/
dst_root = Path("/content/drive/MyDrive/1Trash/Yolov8L/V0")

dst_root.mkdir(parents=True, exist_ok=True)

dst = dst_root / src.name

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print(f"Copied folder to: {dst}")

Copied folder to: /content/drive/MyDrive/1Trash/Yolov8L/V0/gt_vs_pred_fold_Video0
